# BERTWSI

Contextual word-sense induction with multilingual BERT target embeddings, local BERT context, and compact context features for per-word clustering.

## Imports and Configuration

In [ ]:
import re
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "bert-base-multilingual-cased"
BERTWSI_LAYER_COUNT = 4
BERTWSI_MAX_LENGTH = 512
BERTWSI_CONTEXT_WINDOW = 24
LEXICAL_SVD_COMPONENTS = 100
TARGET_VECTOR_WEIGHT = 0.5
LOCAL_CONTEXT_VECTOR_WEIGHT = 0.8
LEXICAL_CONTEXT_VECTOR_WEIGHT = 2.0
KMEANS_N_INIT = 30
RANDOM_STATES = [42, 7, 13, 21, 100]
VALID_FRAC = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(RANDOM_STATES[0])
np.random.seed(RANDOM_STATES[0])

print(f"Device: {DEVICE}")

## Load Data

In [ ]:
df = pd.read_csv("data/train/train.csv", sep="	")
df = df.drop(columns=["predict_sense_id"], errors="ignore")

required_columns = {"context_id", "word", "gold_sense_id", "positions", "context"}
missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"Rows: {len(df)}")
print(f"Words: {df['word'].nunique()}")
print(f"Missing positions: {df['positions'].isna().sum()}")
df.head()

## Load BERT Encoder

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if not getattr(tokenizer, "is_fast", False):
    raise ValueError("BERTWSI requires a fast tokenizer for offset mapping.")

bert_model = AutoModel.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
).to(DEVICE)
bert_model.eval()

{
    "device": DEVICE,
    "model": MODEL_NAME,
    "hidden_size": bert_model.config.hidden_size,
    "pooled_layers": BERTWSI_LAYER_COUNT,
    "max_length": BERTWSI_MAX_LENGTH,
}

## BERTWSI Validation Experiment

In [ ]:
target_token_pattern = re.compile(r"[A-Za-z\u0410-\u042f\u0430-\u044f\u0401\u0451-]+")
resolved_positions = []

for row in df.itertuples():
    position_value = None
    positions_str = "" if pd.isna(row.positions) else str(row.positions)

    for chunk in positions_str.split(","):
        chunk = chunk.strip()
        if "-" not in chunk:
            continue
        start_text, end_text = chunk.split("-", 1)
        try:
            start_char = int(start_text)
            end_char = int(end_text)
        except ValueError:
            continue

        if start_char < end_char:
            position_value = f"{start_char}-{end_char}"
            break

    if position_value is None:
        text_lower = str(row.context).lower()
        word_lower = str(row.word).lower()
        exact_matches = [match.span() for match in re.finditer(re.escape(word_lower), text_lower)]
        if len(exact_matches) == 1:
            start_char, end_char = exact_matches[0]
            position_value = f"{start_char}-{end_char}"

    if position_value is None:
        word_lower = str(row.word).lower()
        token_matches = []

        for match in target_token_pattern.finditer(str(row.context)):
            token = match.group(0)
            prefix_len = 0

            for token_char, word_char in zip(token.lower(), word_lower):
                if token_char != word_char:
                    break
                prefix_len += 1

            min_prefix = max(3, min(len(token), len(word_lower)) - 2)
            if prefix_len >= min_prefix:
                token_matches.append(match.span())

        if len(token_matches) == 1:
            start_char, end_char = token_matches[0]
            position_value = f"{start_char}-{end_char}"

    resolved_positions.append(position_value)

bert_df = df.copy()
bert_df["resolved_positions"] = resolved_positions
unresolved_df = bert_df[bert_df["resolved_positions"].isna()][["context_id", "word", "context"]].copy()
bert_df = bert_df[bert_df["resolved_positions"].notna()].copy()
bert_df["positions"] = bert_df["resolved_positions"]
bert_df = bert_df.drop(columns=["resolved_positions"])
bert_df = bert_df.reset_index().rename(columns={"index": "source_index"})
bert_df["vector_row"] = np.arange(len(bert_df))

print(f"Resolved rows: {len(bert_df)}")
print(f"Unresolved rows: {len(unresolved_df)}")

target_vectors = []
local_context_vectors = []
offset_fallbacks = 0

for row in tqdm(bert_df.itertuples(index=False), total=len(bert_df), desc="Building BERT vectors"):
    start_char = None
    end_char = None

    for chunk in str(row.positions).split(","):
        chunk = chunk.strip()
        if "-" not in chunk:
            continue
        start_text, end_text = chunk.split("-", 1)
        try:
            start_char = int(start_text)
            end_char = int(end_text)
            break
        except ValueError:
            continue

    if start_char is None or end_char is None:
        raise ValueError(f"Cannot parse target position: {row.positions}")

    encoded = tokenizer(
        str(row.context),
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=BERTWSI_MAX_LENGTH,
    )
    offset_mapping = encoded.pop("offset_mapping")[0].tolist()
    model_inputs = {key: value.to(DEVICE) for key, value in encoded.items()}

    with torch.no_grad():
        outputs = bert_model(**model_inputs, output_hidden_states=True, return_dict=True)

    hidden_layers = outputs.hidden_states[1:]
    layer_window = min(BERTWSI_LAYER_COUNT, len(hidden_layers))
    hidden = torch.stack(hidden_layers[-layer_window:], dim=0).mean(dim=0)[0]

    target_token_indices = []
    content_token_indices = []

    for token_idx, (token_start, token_end) in enumerate(offset_mapping):
        if token_end <= token_start:
            continue

        content_token_indices.append(token_idx)
        if token_end > start_char and token_start < end_char:
            target_token_indices.append(token_idx)

    if len(target_token_indices) == 0:
        target_token_indices = [1]
        offset_fallbacks += 1

    target_center = int(round(float(np.mean(target_token_indices))))
    local_token_indices = [
        token_idx
        for token_idx in content_token_indices
        if token_idx not in target_token_indices and abs(token_idx - target_center) <= BERTWSI_CONTEXT_WINDOW
    ]

    if len(local_token_indices) == 0:
        local_token_indices = [token_idx for token_idx in content_token_indices if token_idx not in target_token_indices]

    if len(local_token_indices) == 0:
        local_token_indices = target_token_indices

    target_vector = hidden[target_token_indices].mean(dim=0)
    local_context_vector = hidden[local_token_indices].mean(dim=0)

    target_vectors.append(target_vector.detach().cpu().numpy())
    local_context_vectors.append(local_context_vector.detach().cpu().numpy())

target_vectors = normalize(np.vstack(target_vectors).astype(np.float32), norm="l2")
local_context_vectors = normalize(np.vstack(local_context_vectors).astype(np.float32), norm="l2")

lexical_vectorizer = TfidfVectorizer(
    lowercase=True,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    max_features=50000,
)
lexical_matrix = lexical_vectorizer.fit_transform(bert_df["context"].fillna("").astype(str))
lexical_components = min(LEXICAL_SVD_COMPONENTS, lexical_matrix.shape[0] - 1, lexical_matrix.shape[1] - 1)

if lexical_components >= 2:
    lexical_vectors = TruncatedSVD(
        n_components=lexical_components,
        random_state=RANDOM_STATES[0],
    ).fit_transform(lexical_matrix)
else:
    lexical_vectors = lexical_matrix.toarray()

lexical_vectors = normalize(lexical_vectors.astype(np.float32), norm="l2")
all_vectors = np.hstack(
    [
        TARGET_VECTOR_WEIGHT * target_vectors,
        LOCAL_CONTEXT_VECTOR_WEIGHT * local_context_vectors,
        LEXICAL_CONTEXT_VECTOR_WEIGHT * lexical_vectors,
    ]
)
all_vectors = normalize(all_vectors.astype(np.float32), norm="l2")

bertwsi_runs = []
bertwsi_word_scores = []
all_source_indices = set(bert_df["source_index"])

for random_state in RANDOM_STATES:
    print(f"=== random_state={random_state} ===")
    rng = np.random.RandomState(random_state)
    train_idx = []
    valid_idx = []

    for _, word_df in df.groupby("word", sort=True):
        for _, sense_df in word_df.groupby("gold_sense_id", sort=True):
            indices = sense_df.index.to_numpy().copy()
            rng.shuffle(indices)

            if len(indices) < 2:
                train_idx.extend(indices.tolist())
                continue

            n_valid = max(1, int(round(len(indices) * VALID_FRAC)))
            n_valid = min(n_valid, len(indices) - 1)

            valid_idx.extend(indices[:n_valid].tolist())
            train_idx.extend(indices[n_valid:].tolist())

    train_idx = sorted(set(train_idx) & all_source_indices)
    valid_idx = sorted(set(valid_idx) & all_source_indices)
    train_bert_df = bert_df[bert_df["source_index"].isin(train_idx)].copy()
    valid_bert_df = bert_df[bert_df["source_index"].isin(valid_idx)].copy()
    all_eval_df = pd.concat(
        [
            train_bert_df.assign(split="train"),
            valid_bert_df.assign(split="valid"),
        ],
        axis=0,
        ignore_index=True,
    )

    predictions = pd.Series(index=valid_bert_df["source_index"], dtype="object")

    for word, word_df in all_eval_df.groupby("word", sort=True):
        vector_rows = word_df["vector_row"].to_numpy()
        word_train_df = train_bert_df[train_bert_df["word"] == word]
        n_clusters = int(word_train_df["gold_sense_id"].astype(str).nunique())
        n_clusters = max(1, min(n_clusters, len(word_df)))

        if len(vector_rows) == 0:
            word_labels = np.array([], dtype=int)
        elif len(vector_rows) == 1 or n_clusters <= 1:
            word_labels = np.zeros(len(vector_rows), dtype=int)
        else:
            clusterer = KMeans(
                n_clusters=n_clusters,
                random_state=random_state,
                n_init=KMEANS_N_INIT,
            )
            word_labels = clusterer.fit_predict(all_vectors[vector_rows])

        word_result = word_df[["source_index", "split"]].copy()
        word_result["prediction"] = word_labels.astype(str)
        valid_result = word_result[word_result["split"] == "valid"]
        predictions.loc[valid_result["source_index"]] = valid_result["prediction"].to_numpy()

    scored_df = valid_bert_df[["source_index", "word", "gold_sense_id"]].copy()
    valid_predictions = predictions.loc[valid_bert_df["source_index"]]
    missing_predictions = valid_predictions.isna().sum()
    if missing_predictions:
        raise ValueError(f"Missing predictions for {missing_predictions} validation rows")

    scored_df["prediction"] = valid_predictions.to_numpy().astype(str)
    scored_df["gold_sense_id"] = scored_df["gold_sense_id"].astype(str)

    run_word_scores = []
    for word, word_df in scored_df.groupby("word", sort=True):
        ari = adjusted_rand_score(word_df["gold_sense_id"], word_df["prediction"])
        run_word_scores.append((word, ari, len(word_df)))

    word_scores_df = pd.DataFrame(run_word_scores, columns=["word", "ari", "n_valid"])
    mean_ari = word_scores_df["ari"].mean()
    word_scores_df["model"] = "bertwsi_context_kmeans"
    word_scores_df["random_state"] = random_state

    run = {
        "model": "bertwsi_context_kmeans",
        "random_state": random_state,
        "ari": mean_ari,
        "train_rows": len(train_idx),
        "valid_rows": len(valid_idx),
        "resolved_rows": len(bert_df),
        "unresolved_rows": len(unresolved_df),
        "offset_fallbacks": offset_fallbacks,
        "feature_dim": all_vectors.shape[1],
    }

    bertwsi_runs.append(run)
    bertwsi_word_scores.append(word_scores_df)
    print(f"ARI: {mean_ari:.4f}")

bertwsi_runs_df = pd.DataFrame(bertwsi_runs)
bertwsi_word_scores_df = pd.concat(bertwsi_word_scores, ignore_index=True)
bertwsi_runs_df

## Summary

In [ ]:
bertwsi_summary_df = (
    bertwsi_runs_df.groupby("model", as_index=False)
    .agg(
        ari=("ari", "mean"),
        ari_std=("ari", "std"),
        runs=("ari", "count"),
        resolved_rows=("resolved_rows", "mean"),
        unresolved_rows=("unresolved_rows", "mean"),
        offset_fallbacks=("offset_fallbacks", "mean"),
        feature_dim=("feature_dim", "mean"),
    )
    .sort_values("ari", ascending=False)
)

bertwsi_summary_df

In [ ]:
bertwsi_word_scores_df.sort_values(["random_state", "ari"], ascending=[True, False]).head(20)